# Data wrangling toolbox: regex, datetime and collections
* real world data is messy - text that doesnt quite match the shape we want, dates in weird formats, and lists/dicts that need extra bookkeeping
* python solves a LOT of this with just the standard library, no pip install needed
* this lesson is about three heavy-hitters: the `re` module (pattern matching in strings), the `datetime` module (dates and times) and the `collections` module (specialized container types)
* note: this is different from lesson 05 which was about reading/writing files, json and csv - here we look at the tools you reach for once the data is already in memory as strings, numbers and python objects

## re - regular expressions
* a regular expression (regex) is a mini-language for describing a PATTERN inside text, so we can search, validate or extract stuff without writing a ton of manual string-slicing code
* python gives us access to regex through the builtin `re` module
* fair warning: regex is extremely powerful but also extremely easy to write in a way that nobody (including future-you) can read anymore - use it, but dont go overboard with a single 200-character pattern if a simple `.split()` or `.startswith()` would do the job too

In [1]:
import re  # the standard library module for regular expressions

text = "call me at 555-1234 or email alex@example.com"

# re.search() scans the WHOLE string and returns the first match anywhere inside it (or None if nothing matches)
match = re.search(r"\d{3}-\d{4}", text)  # \d means "a single digit", {3} means "exactly 3 of those"
print("search result:", match)
print("the matched text itself:", match.group())  # .group() (no argument) gives us the full matched substring

# re.match() only checks for a match at the very START of the string - it does NOT scan the whole thing like search() does
print("match() at start:", re.match(r"call", text))   # "call" is the first word --> matches
print("match() for digits:", re.match(r"\d{3}-\d{4}", text))  # the string does not START with digits --> None, even though search() found it above

# re.fullmatch() is the strictest of the three - the ENTIRE string has to match the pattern from start to end, not just part of it
print("fullmatch on phone number:", re.fullmatch(r"\d{3}-\d{4}", "555-1234"))       # the whole string is exactly the pattern --> matches
print("fullmatch on longer text:", re.fullmatch(r"\d{3}-\d{4}", text))              # text has more stuff around it --> None

# rule of thumb: search = "is this pattern ANYWHERE in the string", match = "does the string START with this pattern",
# fullmatch = "does the ENTIRE string look exactly like this pattern" (very useful for validating user input like a full phone number or postcode)

search result: <re.Match object; span=(11, 19), match='555-1234'>
the matched text itself: 555-1234
match() at start: <re.Match object; span=(0, 4), match='call'>
match() for digits: None
fullmatch on phone number: <re.Match object; span=(0, 8), match='555-1234'>
fullmatch on longer text: None


In [2]:
# capturing groups: put part of a pattern in "()" to grab just that piece out of the match, instead of the whole match
phone_pattern = r"(\d{3})-(\d{4})"  # two groups: area code, and the rest of the number
match = re.search(phone_pattern, text)

print("full match:", match.group())     # group() or group(0) --> the whole match
print("group 1 (area code):", match.group(1))
print("group 2 (rest):", match.group(2))
print("all groups as a tuple:", match.groups())

# named groups with (?P<name>...) - same idea, but we can access them by a readable name instead of a position number
# this is a big readability win once you have more than 2-3 groups, positions get confusing fast
named_pattern = r"(?P<area>\d{3})-(?P<rest>\d{4})"
match = re.search(named_pattern, text)
print("named group 'area':", match.group("area"))
print("named group 'rest':", match.group("rest"))
print("as a dict:", match.groupdict())  # .groupdict() turns all named groups into a dict in one go, very handy

full match: 555-1234
group 1 (area code): 555
group 2 (rest): 1234
all groups as a tuple: ('555', '1234')
named group 'area': 555
named group 'rest': 1234
as a dict: {'area': '555', 'rest': '1234'}


In [3]:
# re.findall() returns ALL non-overlapping matches in the string as a list, not just the first one like search()
sentence = "prices went from 10 to 25 and then down to 15 dollars"
all_numbers = re.findall(r"\d+", sentence)  # \d+ means "one or more digits in a row"
print("all numbers found:", all_numbers)  # note: findall gives us STRINGS, not ints - we would still need int() to do math with them

# re.sub() searches for a pattern and SUBSTITUTES it with a replacement string, returning the new string (original is untouched)
censored = re.sub(r"\d+", "[REDACTED]", sentence)
print("censored:", censored)

# re.sub() can also use captured groups in the replacement with backreferences (\1, \2, ...)
# example: swap "firstname lastname" into "lastname, firstname"
name_swapped = re.sub(r"(\w+) (\w+)", r"\2, \1", "Alex Miller")
print("swapped name:", name_swapped)

all numbers found: ['10', '25', '15']
censored: prices went from [REDACTED] to [REDACTED] and then down to [REDACTED] dollars
swapped name: Miller, Alex


In [4]:
# re.compile() pre-compiles a pattern into a reusable pattern object
# why bother? if we run the SAME pattern many times (e.g. inside a loop over thousands of lines), compiling it once upfront is faster
# than letting re.search()/re.findall() re-compile the pattern string every single time internally
email_pattern = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")  # a decent (not perfect!) pattern for email-like text

log_text = """
support request from john.doe@example.com about invoice #4471
cc: billing-team@company.co.uk
please ignore the noreply@localhost address, its just a test entry
"""

# once compiled, we call .findall()/.search()/.sub() directly ON the pattern object instead of passing the pattern string every time
found_emails = email_pattern.findall(log_text)
print("emails found in the log:", found_emails)

# quick reminder on readability: the pattern above already looks a bit cryptic even though its "just" an email matcher
# real, fully RFC-compliant email validation regex is famously huge and unreadable - for anything critical, prefer a well tested library
# or a simpler "good enough" check plus an actual confirmation email, rather than trying to out-regex the entire email spec yourself

emails found in the log: ['john.doe@example.com', 'billing-team@company.co.uk']


## datetime - dates, times and durations
* the `datetime` module is the standard library toolbox for anything involving calendar dates and clock times
* it gives us 4 main types: `date` (just a calendar date), `time` (just a clock time, no date), `datetime` (date + time combined) and `timedelta` (a DURATION/difference between two points in time)

In [5]:
from datetime import date, time, datetime, timedelta

# creating a plain date - just year, month, day, no time-of-day information at all
birthday = date(1995, 7, 21)
print("a date:", birthday)

# creating a plain time - just hour, minute, second, no date attached to it
lunch_time = time(12, 30, 0)
print("a time:", lunch_time)

# creating a full datetime - the combination of both
meeting = datetime(2026, 9, 26, 14, 0, 0)  # year, month, day, hour, minute, second
print("a datetime:", meeting)

# .today() / .now() give us the CURRENT moment, super common in real code (timestamps, logging, "last updated" fields, ...)
print("date.today():", date.today())         # just todays date, no time
print("datetime.now():", datetime.now())      # todays date AND the current clock time

a date: 1995-07-21
a time: 12:30:00
a datetime: 2026-09-26 14:00:00
date.today(): 2026-09-26
datetime.now(): 2026-09-26 10:52:37.876501


In [6]:
# timedelta represents a DURATION - a span of time, not a fixed point in time
one_week = timedelta(days=7)
print("one week as timedelta:", one_week)

# we can do arithmetic directly with dates/datetimes and timedeltas - this is the main reason timedelta exists
today = date.today()
next_week = today + one_week          # adding a duration to a date gives us a new date
print("today:", today, "--> next week:", next_week)

three_days_ago = today - timedelta(days=3)  # subtracting works the same way
print("three days ago:", three_days_ago)

# subtracting two dates from EACH OTHER gives us back a timedelta (the difference between them)
difference = next_week - today
print("difference between today and next week:", difference, "-->", difference.days, "days")

# timedelta also supports bigger units directly (weeks) and combining several units at once
two_and_a_half_weeks = timedelta(weeks=2, days=3, hours=12)
print("a more complex timedelta:", two_and_a_half_weeks)

one week as timedelta: 7 days, 0:00:00
today: 2026-09-26 --> next week: 2026-10-03
three days ago: 2026-09-23
difference between today and next week: 7 days, 0:00:00 --> 7 days
a more complex timedelta: 17 days, 12:00:00


In [7]:
# .strftime() formats a date/datetime object INTO a human readable string, using format codes like %Y %m %d %H %M %S
now = datetime.now()
print("iso-ish:", now.strftime("%Y-%m-%d %H:%M:%S"))   # %Y = 4-digit year, %m = month, %d = day, %H/%M/%S = hour/minute/second
print("us style:", now.strftime("%m/%d/%Y"))
print("readable:", now.strftime("%A, %d %B %Y"))       # %A = full weekday name, %B = full month name

# .strptime() does the exact OPPOSITE - it PARSES a string back INTO a datetime object, using the same format codes
# the format string has to match the input string exactly, otherwise it raises a ValueError
parsed = datetime.strptime("26.09.2026", "%d.%m.%Y")   # a typical german date format: day.month.year
print("parsed back into a datetime object:", parsed, type(parsed))

# comparing two dates/datetimes works with the normal comparison operators, exactly like numbers
deadline = date(2026, 12, 31)
print("is today before the deadline?", today < deadline)
print("days left until the deadline:", (deadline - today).days)

iso-ish: 2026-09-26 10:52:37
us style: 09/26/2026
readable: Saturday, 26 September 2026
parsed back into a datetime object: 2026-09-26 00:00:00 <class 'datetime.datetime'>
is today before the deadline? True
days left until the deadline: 96


In [8]:
from datetime import timezone

# an important gotcha: datetime.now() (without arguments) gives us a "naive" datetime - it has NO idea which timezone it belongs to
# it just represents "some wall clock reading somewhere", python trusts you to know what that means
naive_now = datetime.now()
print("naive datetime (no timezone attached):", naive_now, "tzinfo:", naive_now.tzinfo)

# datetime.now(timezone.utc) gives us an "aware" datetime - it explicitly knows it represents UTC time
aware_now_utc = datetime.now(timezone.utc)
print("timezone-aware datetime (UTC):", aware_now_utc, "tzinfo:", aware_now_utc.tzinfo)

# why does this matter? comparing/subtracting a naive datetime with an aware one raises a TypeError - python refuses to guess
# for any application that deals with users in different countries/timezones (scheduling, logging servers, ...) you want AWARE datetimes
# going deeper into real timezone handling (daylight saving, "Europe/Berlin" vs "America/New_York", ...) needs the `zoneinfo` module
# (standard library since python 3.9) or the third-party `pytz`/`dateutil` packages - out of scope for this lesson, just know the concept exists

naive datetime (no timezone attached): 2026-09-26 10:52:37.900546 tzinfo: None
timezone-aware datetime (UTC): 2026-09-26 10:52:37.900701+00:00 tzinfo: UTC


## collections.Counter - counting things fast
* `Counter` is a specialized dict subclass built for exactly one job: counting how often things appear in an iterable
* instead of writing manual "if key in dict, increment, else set to 1" logic, `Counter` just does it for us in one line

In [9]:
from collections import Counter

sentence = "the quick brown fox jumps over the lazy dog the fox runs the fastest"
words = sentence.split()  # .split() with no argument splits on any whitespace, gives us a list of words

word_counts = Counter(words)  # Counter takes any iterable and counts how often each element shows up
print("all counts:", word_counts)
print("how often does 'the' appear?", word_counts["the"])  # works just like a normal dict lookup
print("how often does a missing word appear?", word_counts["banana"])  # a normal dict would raise KeyError here, Counter just returns 0!

# .most_common(n) gives us the n most frequent elements, sorted from most to least common, as a list of (element, count) tuples
print("3 most common words:", word_counts.most_common(3))
print("single most common word:", word_counts.most_common(1))

all counts: Counter({'the': 4, 'fox': 2, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1, 'runs': 1, 'fastest': 1})
how often does 'the' appear? 4
how often does a missing word appear? 0
3 most common words: [('the', 4), ('fox', 2), ('quick', 1)]
single most common word: [('the', 4)]


## collections.defaultdict - never worry about missing keys again
* a very classic problem: we want to build a dict where each key maps to a LIST of things, by appending to that list as we go
* the plain dict version needs a manual check every single time ("does this key already exist? if not, create an empty list first")
* `defaultdict` removes that boilerplate completely by auto-creating a default value the first time a new key is used

In [10]:
# the plain-dict version - notice the manual "if key not in dict" check we need EVERY time before appending
students_by_grade_plain = {}
entries = [("A", "Alex"), ("B", "Mia"), ("A", "Sam"), ("C", "Jo"), ("B", "Lee")]

for grade, name in entries:
    if grade not in students_by_grade_plain:   # without this check, students_by_grade_plain[grade].append(name) would raise KeyError
        students_by_grade_plain[grade] = []     # so we manually create an empty list the first time we see this grade
    students_by_grade_plain[grade].append(name)

print("plain dict version:", students_by_grade_plain)

plain dict version: {'A': ['Alex', 'Sam'], 'B': ['Mia', 'Lee'], 'C': ['Jo']}


In [11]:
from collections import defaultdict

# the defaultdict version - we tell it "if a key doesnt exist yet, create it using list() (an empty list)" and never worry again
students_by_grade = defaultdict(list)  # list is the "factory function" - called with no arguments whenever a missing key is accessed

for grade, name in entries:
    students_by_grade[grade].append(name)  # no manual check needed anymore, defaultdict handles the "create empty list first" step for us

print("defaultdict version:", dict(students_by_grade))  # wrapping in dict() just for a cleaner print, defaultdict itself works fine as-is

# defaultdict works with ANY zero-argument callable as the factory, not just list
# defaultdict(int) is another very common pattern - handy for counting (though Counter above is usually the better tool for pure counting)
counts = defaultdict(int)  # int() with no arguments returns 0, so every new key starts at 0
for grade, name in entries:
    counts[grade] += 1
print("counting with defaultdict(int):", dict(counts))

defaultdict version: {'A': ['Alex', 'Sam'], 'B': ['Mia', 'Lee'], 'C': ['Jo']}
counting with defaultdict(int): {'A': 2, 'B': 2, 'C': 1}


## collections.namedtuple - a lightweight record type
* sometimes we just want a small, immutable "bag of fields" - like a Point(x, y) or a Person(name, age) - without writing a full class
* `namedtuple` is a factory FUNCTION that builds a new tuple-based class for us, with named fields we can access by `.fieldname` instead of only `[0]`, `[1]`, ...
* note: lesson 10 goes deeper and compares this to `@dataclass` and `typing.NamedTuple`, which are more modern alternatives - here we just keep it to the classic, oldest form

In [12]:
from collections import namedtuple

# namedtuple(typename, [list of field names]) returns a brand new class, which we then use just like any other class
Point = namedtuple("Point", ["x", "y"])

p1 = Point(3, 4)
print("a point:", p1)
print("accessing fields by name:", p1.x, p1.y)      # much more readable than p1[0], p1[1]
print("still works like a normal tuple too:", p1[0], p1[1])
print("unpacking still works:", *p1)

# namedtuples are IMMUTABLE, just like regular tuples - we cant change a field after creation
try:
    p1.x = 99
except AttributeError as error:
    print("cant modify a namedtuple field:", error)

# to get a "changed copy", namedtuple gives us ._replace() which returns a NEW instance with some fields swapped out
p2 = p1._replace(x=99)
print("original untouched:", p1, "-- new copy:", p2)

a point: Point(x=3, y=4)
accessing fields by name: 3 4
still works like a normal tuple too: 3 4
unpacking still works: 3 4
cant modify a namedtuple field: can't set attribute
original untouched: Point(x=3, y=4) -- new copy: Point(x=99, y=4)


## collections.deque - a double-ended queue
* a `deque` (pronounced "deck", short for "double-ended queue") is a list-like container that is optimized for adding/removing items from BOTH ends
* the key difference to a plain `list`: `list.insert(0, x)` (insert at the front) is O(n) - it has to shift every single existing element one slot to the right
* `deque.appendleft(x)` is O(1) - constant time, no matter how big the deque already is, because a deque is internally structured to make front-operations cheap too

In [13]:
from collections import deque

queue = deque(["b", "c"])  # deque can be created directly from any iterable, just like list()
print("starting deque:", queue)

queue.append("d")       # add to the RIGHT end - O(1), same as list.append()
queue.appendleft("a")   # add to the LEFT end - O(1) for deque, but list.insert(0, x) would be O(n)!
print("after append/appendleft:", queue)

last = queue.pop()          # remove and return from the RIGHT end - O(1)
first = queue.popleft()     # remove and return from the LEFT end - O(1)
print("popped from the right:", last, "-- popped from the left:", first)
print("remaining deque:", queue)

# why does the O(n) vs O(1) difference actually matter? imagine processing a queue of 100,000 items, always removing from the front
# with a list: EVERY single .pop(0) call has to shift up to 100,000 remaining elements one position - that adds up to a huge slowdown
# with a deque: EVERY .popleft() call is equally fast no matter the size, because deque is built (as a doubly linked list of blocks) exactly for this
# rule of thumb: use deque whenever you need a queue/fifo structure or repeatedly touch BOTH ends of a collection, use list when you mostly work at the end

starting deque: deque(['b', 'c'])
after append/appendleft: deque(['a', 'b', 'c', 'd'])
popped from the right: d -- popped from the left: a
remaining deque: deque(['b', 'c'])


## collections.OrderedDict - a quick, brief mention
* historically, `OrderedDict` existed because plain python `dict` did NOT guarantee any particular order of its keys
* since python 3.7, regular `dict` preserves INSERTION order as an official language guarantee - so for "just remembering insertion order", a plain `dict` is now enough
* `OrderedDict` still exists and is occasionally useful for its extra `.move_to_end()` method, or simply to make the "order matters here" intent explicit and obvious to readers/older code

In [14]:
from collections import OrderedDict

regular_dict = {"first": 1, "second": 2, "third": 3}
print("regular dict already keeps insertion order since python 3.7:", regular_dict)

ordered = OrderedDict(first=1, second=2, third=3)
ordered.move_to_end("first")  # moves an existing key to the end - a plain dict has no built-in shortcut for this
print("after move_to_end('first'):", ordered)

regular dict already keeps insertion order since python 3.7: {'first': 1, 'second': 2, 'third': 3}
after move_to_end('first'): OrderedDict([('second', 2), ('third', 3), ('first', 1)])


## whats next: numpy and pandas
* everything in this lesson is pure standard library - great for everyday text/date/collection wrangling on small to medium amounts of data
* once you work with SERIOUS numeric data (big arrays of numbers, matrix math, statistics) or spreadsheet-like TABULAR data (rows and columns, filtering, grouping, joining datasets), the wider python ecosystem has two extremely popular third-party libraries: `numpy` (fast, memory-efficient numeric arrays and vectorized math, avoiding slow python-level for-loops over numbers) and `pandas` (a "DataFrame" object thats basically a spreadsheet/database table you can filter, group, merge and analyze with a few lines of code)
* both need an extra `pip install numpy pandas` step since they are NOT part of the standard library - we wont install or use them in this lesson (keeping things dependency-free), just keep them in mind as the natural next step once plain python containers start to feel limiting for data-heavy work

## quick comparison to other languages
* pythons `re` module and its pattern syntax (`\d`, `\w`, capturing groups `()`, named groups `(?P<name>...)`) is nearly identical to JavaScripts `RegExp`/`/pattern/flags` literals, Javas `java.util.regex.Pattern`, and pretty much every other mainstream language - regex is one of the most universal "mini-languages" in all of programming, learn it once and reuse it almost everywhere
* pythons `datetime` module maps conceptually to JavaScripts `Date` object (and the newer `Temporal` proposal), Javas `java.time` package (`LocalDate`, `LocalDateTime`, `Duration`), and C#s `DateTime`/`DateTimeOffset` types - every mainstream language has separate concepts for "a point in time" and "a duration/difference", just under different names
* `Counter` roughly corresponds to manually counting with a `HashMap`/`Map` in Java or C++ (`std::unordered_map`), or `Map.getOrDefault(key, 0) + 1` in modern Java/JS - python just ships the counting logic pre-built
* `defaultdict` is similar in spirit to Javas `Map.computeIfAbsent(key, k -> new ArrayList<>())` or JavaScripts manual `if (!map.has(key)) map.set(key, [])` pattern - again, python bakes the "create default if missing" behaviour directly into the container
* `deque` matches C++s `std::deque` and Javas `ArrayDeque` almost one-to-one (both are also O(1) at either end) - plain JavaScript arrays technically support `.unshift()`/`.shift()` for front operations, but (like pythons list) they are generally much less efficient for that than a real deque structure

## Exercises
1. given the text `data = "order #123 shipped, 4 items, total $567.89, tracking 88-11"`, use `re.findall` to extract every number (as strings) that appears anywhere in it
2. given `words = ["red", "blue", "red", "green", "blue", "red", "yellow", "blue"]`, use `Counter` to find the 3 most common words and how often each occurs
3. given `entries = [("fruit", "apple"), ("veggie", "carrot"), ("fruit", "banana"), ("veggie", "potato"), ("fruit", "cherry")]`, use `defaultdict(list)` to group them into a dict mapping each category to a list of its items
4. given `target_date = date(2027, 1, 1)`, compute and print how many days are left until that date, counting from todays date (use `datetime`/`timedelta`)

In [15]:
# TODO: solve exercise 1 here (re.findall for all numbers)


# TODO: solve exercise 2 here (Counter, 3 most common words)


# TODO: solve exercise 3 here (defaultdict(list) grouping by category)


# TODO: solve exercise 4 here (days left until target_date)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [16]:
import re
from collections import Counter, defaultdict
from datetime import date

# sample solution 1
data = "order #123 shipped, 4 items, total $567.89, tracking 88-11"
numbers = re.findall(r"\d+", data)  # \d+ grabs one-or-more consecutive digits at a time, so "567.89" comes back as two pieces "567" and "89"
print("all numbers found:", numbers)

# sample solution 2
words = ["red", "blue", "red", "green", "blue", "red", "yellow", "blue"]
word_counts = Counter(words)
print("3 most common words:", word_counts.most_common(3))

# sample solution 3
entries = [("fruit", "apple"), ("veggie", "carrot"), ("fruit", "banana"), ("veggie", "potato"), ("fruit", "cherry")]
grouped = defaultdict(list)
for category, item in entries:
    grouped[category].append(item)
print("grouped by category:", dict(grouped))

# sample solution 4
target_date = date(2027, 1, 1)
today = date.today()
days_left = (target_date - today).days
print("days left until target date:", days_left)

all numbers found: ['123', '4', '567', '89', '88', '11']
3 most common words: [('red', 3), ('blue', 3), ('green', 1)]
grouped by category: {'fruit': ['apple', 'banana', 'cherry'], 'veggie': ['carrot', 'potato']}
days left until target date: 97


* congratulation you finished this lesson

## what we learned
* the `re` module - `search` vs `match` vs `fullmatch`, `findall`, `sub`, capturing groups (positional and named), and reusing a `re.compile()` pattern
* the `datetime` module - `date`, `time`, `datetime`, `timedelta`, doing date arithmetic, formatting with `.strftime()`, parsing with `.strptime()`, comparing dates, and naive vs timezone-aware datetimes
* `collections.Counter` for fast counting and `.most_common()`
* `collections.defaultdict` to avoid manual "check if key exists" boilerplate when building dicts of lists
* `collections.namedtuple` as a lightweight, immutable alternative to a full class for small records
* `collections.deque` for efficient O(1) operations at both ends, and why that beats a plain `list` for front-heavy workloads
* `collections.OrderedDict` and why plain `dict` already covers most of its historical use case since python 3.7
* a quick pointer towards `numpy`/`pandas` for heavier numeric/tabular data work later on